## Importing necessary packages

In [ ]:
import os
import random
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from mtcnn import MTCNN
from tensorflow.keras import layers, callbacks, mixed_precision
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision enabled")

print("GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

## Setting up dataset paths and output directories

In [ ]:
# Celeb-DF v2
CELEB_REAL_DIR = "//kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-real"
CELEB_FAKE_DIR = "/kaggle/input/datasets/reubensuju/celeb-df-v2/Celeb-synthesis"

# FaceForensics++ C23
FF_BASE = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"

FF_REAL_DIR = f"{FF_BASE}/original"
FF_DEEPFAKES_DIR = f"{FF_BASE}/Deepfakes"
FF_FACE2FACE_DIR = f"{FF_BASE}/Face2Face"
FF_FACESWAP_DIR = f"{FF_BASE}/FaceSwap"
FF_FACESHIFTER_DIR = f"{FF_BASE}/FaceShifter"
FF_NEURALTEXTURES_DIR = f"{FF_BASE}/NeuralTextures"
FF_DFD_DIR = f"{FF_BASE}/DeepFakeDetection"

REAL_SOURCES = [ FF_REAL_DIR, CELEB_REAL_DIR ]

FAKE_SOURCES = [ FF_DEEPFAKES_DIR, FF_FACESWAP_DIR, CELEB_FAKE_DIR ]

BASE_OUTPUT = "/kaggle/working/dataset"
TRAIN_REAL = f"{BASE_OUTPUT}/train/real"
TRAIN_FAKE = f"{BASE_OUTPUT}/train/fake"
VAL_REAL = f"{BASE_OUTPUT}/val/real"
VAL_FAKE = f"{BASE_OUTPUT}/val/fake"
TEST_REAL = f"{BASE_OUTPUT}/test/real"
TEST_FAKE = f"{BASE_OUTPUT}/test/fake"

ALL_FOLDERS = [ TRAIN_REAL, TRAIN_FAKE, VAL_REAL, VAL_FAKE, TEST_REAL, TEST_FAKE ]

for folder in ALL_FOLDERS:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully")

## Face Extraction Function using MTCNN and setting up the extraction process

In [ ]:
def already_extracted():
    return len(os.listdir(TRAIN_REAL)) > 100 and len(os.listdir(TRAIN_FAKE)) > 100

if already_extracted():
    print("Frames already extracted → skipping extraction")
    SKIP_EXTRACTION = True
else:
    print("No cached frames found → starting extraction")
    SKIP_EXTRACTION = False

In [ ]:
# Initialize MTCNN detector
detector = MTCNN()

def extract_face(frame):
    # MTCNN expects RGB images
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect faces
    results = detector.detect_faces(rgb_frame)

    if not results:
        return None

    largest_face = None
    max_area = 0

    # Find the largest face if multiple people are in the frame
    for result in results:
        x, y, w, h = result['box']

        # MTCNN can sometimes return negative coordinates for boxes that go off-screen
        x, y = max(0, x), max(0, y)

        area = w * h
        if area > max_area:
            max_area = area
            largest_face = (x, y, w, h)

    if largest_face is None:
        return None

    x, y, w, h = largest_face

    # Ensure bounding box doesn't exceed image dimensions
    ih, iw, _ = frame.shape
    w, h = min(iw - x, w), min(ih - y, h)

    face = frame[y:y+h, x:x+w]

    # Filter out extremely small or corrupted face crops
    if face.shape[0] < 50 or face.shape[1] < 50:
        return None

    return face

## Function to get all video paths from a directory (with a limit)

In [ ]:
def get_all_video_paths(root_dir, max_videos=500):
    video_paths = []

    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith((".mp4", ".avi", ".mov", ".mkv")):
                video_paths.append(os.path.join(root, file))

    return video_paths[:max_videos]

def split_videos(video_paths):
    train_videos, temp_videos = train_test_split( video_paths, test_size=0.2, random_state=42 )

    val_videos, test_videos = train_test_split( temp_videos, test_size=0.5, random_state=42 )

    return train_videos, val_videos, test_videos

def process_single_video(video_path, output_dir, label, max_frames_per_video=12):
    video_name = os.path.basename(video_path)
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    if fps == 0: fps = 30

    frame_interval = max(1, fps)
    frame_count = 0
    saved_count = 0

    success, frame = cap.read()
    while success and saved_count < max_frames_per_video:
        if frame_count % frame_interval == 0:
            face = extract_face(frame)

            if face is not None:
                face = cv2.resize(face, (224, 224))
                filename = f"{label}_{video_name}_{saved_count}.jpg"
                save_path = os.path.join(output_dir, filename)
                cv2.imwrite(save_path, face)
                saved_count += 1

        success, frame = cap.read()
        frame_count += 1

    cap.release()
    return saved_count

def extract_frames_from_video_list(video_list, output_dir, label, max_frames_per_video=12):
    print(f"Processing {len(video_list)} videos → {output_dir}")
    total_saved = 0

    # This is the crucial part that speeds up the execution
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        futures = [
            executor.submit(process_single_video, video_path, output_dir, label, max_frames_per_video)
            for video_path in video_list
        ]

        for future in tqdm(futures, total=len(video_list), desc=f"Extracting {label}"):
            total_saved += future.result()

    print(f"Saved {total_saved} frames → {output_dir}")

## Extracting frames from REAL and FAKE videos with specified limits

In [ ]:
if not SKIP_EXTRACTION:

    print("\nProcessing REAL videos...\n")

    all_real_videos = []
    for source in REAL_SOURCES:
        videos = get_all_video_paths( source, max_videos=400 )
        all_real_videos.extend(videos)

    train_real, val_real, test_real = split_videos( all_real_videos )

    extract_frames_from_video_list( train_real, TRAIN_REAL, "real_train" )
    extract_frames_from_video_list( val_real, VAL_REAL, "real_val" )
    extract_frames_from_video_list( test_real, TEST_REAL, "real_test" )

    print("\nProcessing FAKE videos...\n")

    all_fake_videos = []
    for source in FAKE_SOURCES:
        videos = get_all_video_paths( source, max_videos=250 )
        all_fake_videos.extend(videos)

    train_fake, val_fake, test_fake = split_videos( all_fake_videos )

    extract_frames_from_video_list( train_fake, TRAIN_FAKE, "fake_train" )
    extract_frames_from_video_list( val_fake, VAL_FAKE, "fake_val" )
    extract_frames_from_video_list( test_fake, TEST_FAKE, "fake_test" )

    print("\nExtraction completed successfully")

print("TRAIN REAL:", len(os.listdir(TRAIN_REAL)))
print("TRAIN FAKE:", len(os.listdir(TRAIN_FAKE)))

print("VAL REAL:", len(os.listdir(VAL_REAL)))
print("VAL FAKE:", len(os.listdir(VAL_FAKE)))

print("TEST REAL:", len(os.listdir(TEST_REAL)))
print("TEST FAKE:", len(os.listdir(TEST_FAKE)))

## Data agumentation and preparing the data for model training

In [ ]:
img_size = (224, 224)
batch_size = 32

data_augmentation = tf.keras.Sequential([ layers.RandomFlip("horizontal"), layers.RandomRotation(0.05), layers.RandomTranslation(0.1, 0.1), layers.RandomContrast(0.2), ])

train_ds = tf.keras.utils.image_dataset_from_directory( f"{BASE_OUTPUT}/train", image_size=img_size, batch_size=batch_size, label_mode="int" )
val_ds = tf.keras.utils.image_dataset_from_directory( f"{BASE_OUTPUT}/val", image_size=img_size, batch_size=batch_size, label_mode="int" )
test_ds = tf.keras.utils.image_dataset_from_directory( f"{BASE_OUTPUT}/test", image_size=img_size, batch_size=batch_size, label_mode="int" )

print("Class Names:", train_ds.class_names)
def prepare(ds, augment=False):
    # Cache to memory BEFORE processing to avoid reading from disk every epoch
    ds = ds.cache()

    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare(train_ds, augment=True) # augment parameter is now just a formality or can be removed
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

print("\nOptimized dataset ready for training.")

## Created the base model before training

In [ ]:
# 1. Initialize the distribution strategy
strategy = tf.distribute.MirroredStrategy()
print(f"Number of GPUs being used: {strategy.num_replicas_in_sync}")

# 2. Open the strategy scope
with strategy.scope():

    base_model = tf.keras.applications.EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling="avg"
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x = base_model(inputs, training=False)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    # CRITICAL FIX: The final layer MUST be forced to float32 to prevent NaN Softmax
    outputs = layers.Dense(2, activation="softmax", dtype="float32")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=1e-3,
            weight_decay=1e-4,
            epsilon=1e-4,  # CRITICAL FIX: Stabilizes float16 division
            clipnorm=1.0   # CRITICAL FIX: Prevents exploding gradients
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


## Setting up callbacks and starting with the first phase of training

In [ ]:
callbacks_list = [
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,  # Reduced from 3
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",  # Changed from val_accuracy
        patience=4,          # Reduced from 6
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        "./video_model.keras",
        monitor="val_loss",  # Changed from val_accuracy
        save_best_only=True
    )
]

class_weights = {
    0: 1.35,   # fake
    1: 1.00    # real
}

print("Starting Phase 1 Training...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_list,
    class_weight=class_weights
    # steps_per_epoch=100,  # <-- Uncomment these if an epoch still takes too long
    # validation_steps=50   # <-- Uncomment these if an epoch still takes too long
)

## Fine tuning the model when required

In [ ]:
print("Starting Fine-Tuning (Phase 2)...")

# Create a dedicated callback list for Phase 2
callbacks_list_phase2 = [
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        "./video_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

class_weights_balanced = {
    0: 1.00,   # fake
    1: 1.00    # real
}

with strategy.scope():
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    # Recompile to reset the optimizer state and use the lower learning rate
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=1e-5,
            weight_decay=1e-5,
            epsilon=1e-4,
            clipnorm=1.0
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks_list_phase2, # <--- USE THE NEW CALLBACKS
    class_weight=class_weights_balanced
)

## For the testing the model later in backend

In [ ]:
model.load_weights("./video_model.keras")

def predict_video(video_path, threshold=0.5):
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30

    frame_interval = max(1, fps // 2) # Extract 2 frames per second
    frame_count = 0
    faces_batch = []

    success, frame = cap.read()

    while success:
        if frame_count % frame_interval == 0:
            face = extract_face(frame)

            if face is not None:
                face = cv2.resize(face, (224, 224))
                face = face.astype(np.float32)
                faces_batch.append(face)

        success, frame = cap.read()
        frame_count += 1

    cap.release()

    if len(faces_batch) == 0:
        return "Error: No faces detected in video."

    batch_array = np.array(faces_batch)

    # Predict in one batch
    preds = model.predict(batch_array, verbose=0)

    # Get the "Fake" probabilities
    fake_probabilities = preds[:, 0]
    average_fake_prob = np.mean(fake_probabilities)

    if average_fake_prob > threshold:
        return f"🚨 FAKE DETECTED (Confidence: {average_fake_prob:.2%})"
    else:
        return f"✅ REAL VIDEO (Fake Probability: {average_fake_prob:.2%})"

# print(predict_video("/kaggle/working/dataset/test/real/real_test_003.mp4_3.jpg"))

## Evaluating the model on test dataset

In [ ]:
model.load_weights("./video_model.keras")

print("Evaluating model on test dataset...\n")

test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

## Set up the classification report for the model

In [ ]:
all_labels = []
all_predictions = []
all_probabilities = []

for images, labels in test_ds:

    preds = model.predict(images, verbose=0)

    predicted_classes = np.argmax(preds, axis=1)

    all_predictions.extend(predicted_classes)
    all_labels.extend(labels.numpy())

    # probability of fake class (index 0 because class_names = ['fake', 'real'])
    all_probabilities.extend(preds[:, 0])

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

In [ ]:
print("\nTrue label distribution:")
unique_true, counts_true = np.unique(all_labels, return_counts=True)
print(dict(zip(unique_true, counts_true)))

print("\nPrediction distribution:")
unique_pred, counts_pred = np.unique(all_predictions, return_counts=True)
print(dict(zip(unique_pred, counts_pred)))

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================\n")
print( classification_report( all_labels, all_predictions, target_names=["fake", "real"], zero_division=0 ) )

final_accuracy = accuracy_score(all_labels, all_predictions)
print(f"\nFinal Accuracy Score: {final_accuracy * 100:.2f}%")

cm = confusion_matrix(all_labels, all_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap( cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred Fake", "Pred Real"], yticklabels=["Actual Fake", "Actual Real"] )
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()